In [10]:
import os
import pickle
import numpy as np
import pandas as pd
def compute_cluster_activation_stats(base_dir="../"):
    """
    Reads ActivationRanges.pkl files from subdirectories (pruning %)
    and computes avg min, avg max, and range per cluster.

    Returns:
    --------
    dict:
        {
          model_name: {
            pruning_pct: {
              'c1': {'avg_min': ..., 'avg_max': ..., 'range': ...},
              'c2': {...},
              'c3': {...}
            }
          }
        }
    """
    cluster_stats = {}

    for percent in os.listdir(base_dir):
        pkl_path = os.path.join(base_dir, percent, "ActivationRanges.pkl")
        if not os.path.isfile(pkl_path):
            continue

        with open(pkl_path, "rb") as f:
            res = pickle.load(f)

        # Initialize accumulators
        mins = {"c1": 0.0, "c2": 0.0, "c3": 0.0}
        maxs = {"c1": 0.0, "c2": 0.0, "c3": 0.0}
        counts = {"c1": 0, "c2": 0, "c3": 0}

        for neuron in res:
            for idx, cname in enumerate(["c1", "c2", "c3"]):
                try:
                    mins[cname] += neuron[idx][0]
                    maxs[cname] += neuron[idx][1]
                    counts[cname] += 1
                except Exception:
                    continue

        # Build stats for this pruning percentage
        pct_stats = {}
        for cname in ["c1", "c2", "c3"]:
            if counts[cname] == 0:
                continue
            avg_min = mins[cname] / counts[cname]
            avg_max = maxs[cname] / counts[cname]
            pct_stats[cname] = {
                "avg_min": avg_min,
                "avg_max": avg_max,
                "range": avg_max - avg_min
            }

        # Store (using a single model key, adjust if needed)
        model = "bert"   # or whatever model name you want
        cluster_stats.setdefault(model, {})[percent] = pct_stats

    return cluster_stats

def create_activation_tables(cluster_stats, output_prefix='activation_table'):
    """
    Create tables with clusters on rows and pruning percentages on columns
    for both average min and average max activations.
    
    Parameters:
    -----------
    cluster_stats : dict
        Nested dictionary from compute_cluster_statistics function
    output_prefix : str
        Prefix for output CSV files
    
    Returns:
    --------
    dict : Dictionary of DataFrames for each model and metric
    """
    
    tables = {}
    
    for model in cluster_stats:
        data = cluster_stats[model]
        pruning_pcts = sorted(data.keys())
        clusters = sorted(list(set([c for pct in data for c in data[pct]])))
        
        # Create table for average min activations
        min_data = []
        for cluster in clusters:
            row = [data[pct][cluster]['avg_min'] if cluster in data[pct] else np.nan 
                   for pct in pruning_pcts]
            min_data.append(row)
        
        df_min = pd.DataFrame(
            min_data,
            index=clusters,
            columns=[f'{pct}%' for pct in pruning_pcts]
        )
        df_min.index.name = 'Cluster'
        
        # Create table for average max activations
        max_data = []
        for cluster in clusters:
            row = [data[pct][cluster]['avg_max'] if cluster in data[pct] else np.nan 
                   for pct in pruning_pcts]
            max_data.append(row)
        
        df_max = pd.DataFrame(
            max_data,
            index=clusters,
            columns=[f'{pct}%' for pct in pruning_pcts]
        )
        df_max.index.name = 'Cluster'
        
        range_data = []
        for cluster in clusters:
            row = [data[pct][cluster]['range'] if cluster in data[pct] else np.nan 
                   for pct in pruning_pcts]
            range_data.append(row)
        
        df_range = pd.DataFrame(
            range_data,
            index=clusters,
            columns=[f'{pct}%' for pct in pruning_pcts]
        )
        df_range.index.name = 'Cluster'
        
        # Store tables
        tables[f'{model}_avg_min'] = df_min
        tables[f'{model}_avg_max'] = df_max
        tables[f'{model}_range'] = df_range
        
        # Save to CSV
        
        print(f"\n{model} - Average Min Activations:")
        print(df_min.to_string(float_format='%.4f'))
        
        print(f"\n{model} - Average Max Activations:")
        print(df_max.to_string(float_format='%.4f'))
        
        print(f"\n{model} - Range Activations:")
        print(df_range.to_string(float_format='%.4f'))
    
    return tables


In [11]:
cluster_stats = compute_cluster_activation_stats("/workspace/CCE_NLI/BERT/exp/CoFi/Run0.25_new/Masks")
tables = create_activation_tables(cluster_stats)



bert - Average Min Activations:
         0.0%Pruned%  0.27243559109531656%Pruned%  0.4451998451203647%Pruned%  0.5874358855616119%Pruned%  0.783612596400868%Pruned%
Cluster                                                                                                                             
c1            0.1151                       0.1200                      0.1478                      0.1974                     0.0001
c2            0.3694                       0.3622                      0.3858                      0.4054                     0.4693
c3            0.5030                       0.4891                      0.5042                      0.5030                     1.0795

bert - Average Max Activations:
         0.0%Pruned%  0.27243559109531656%Pruned%  0.4451998451203647%Pruned%  0.5874358855616119%Pruned%  0.783612596400868%Pruned%
Cluster                                                                                                                             
c1 